### General definitions

In [ ]:
class FasterAnalyzer:
    # uses Armenian Uniparser, adds caching for speed

    def __init__(self, parser):
        self.parser = parser
        self._cache = {}

    def analyze_words(self, word):
        # analyzes a word, returns analyses

        if word in self._cache:
        # check if already processed and in cache
            return self._cache[word]

        else:
            # process with analyzer
            analyses = self.parser.analyze_words(word, format='json')

            # add to cache
            self._cache[word] = analyses
            return analyses

In [ ]:
def arm_remove_punct(token):
    from string import punctuation
    # string.punctuation + "Cyrillic" quotation marks + Armenian full stop +
    # + Armenian comma ՝ + ` (found in OCR in place of ՝)
    token = token.strip(punctuation + "«»։՝`")
    # Armenian possibly word-internal punctuation symbols
    token = token.replace('՛', '')
    token = token.replace('՜', '')
    token = token.replace('՞', '')
    return token

### Data

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_pickle('data.pkl')
df

,title,text,date,source,genre,tokens_number
0,ԴՊՐՈՑԻ ՃԱՆՓԱՆ,ԴՊՐՈՑԻ ՃԱՆՓԱՆ \n(Թոփտիի խոսվածքով) \n\nԲուզլավ...,1968,Atamanyan_2006,poem,83
1,ՄԵՐ «ԸՆԿԵՐԸ»,"ՄԵՐ «ԸՆԿԵՐԸ» \n\nԴասարանին դուռը բացվեցավ, Երկ...",1999,Atamanyan_2006,poem,66
2,ՄԵՐ ԽԱՄՈՒԹԻ ՃԱՆՓԱՆ,ՄԵՐ ԽԱՄՈՒԹԻ ՃԱՆՓԱՆ \n(Թոփտիի խոսվածքով) \n\nՄե...,1971,Atamanyan_2006,poem,93
3,ՄԱՇԻՆԸ,"ՄԱՇԻՆԸ \n\nԳասպար, Խաթուն միասին, Քաղաք էշտալ ...",1971,Atamanyan_2006,poem,98
4,ԹՈՓՏԻՆ ԵՎ ԹՈՓՑԻԿԸ,ԹՈՓՏԻՆ ԵՎ ԹՈՓՑԻԿԸ \n\nԻնչքան տարուք դուն կառնո...,1973,Atamanyan_2006,poem,153
...,...,...,...,...,...,...
163,ԵՐԳ ՔԵՖԵԻ ՄԱՍԻՆ,"ԵՐԳ ՔԵՖԵԻ ՄԱՍԻՆ\n\nՔէֆէ քաղաք, բայձառ քաղաք, \...",,Dagldiyan_2023,unknown,69
164,ՍԱՆԴՌԻ ԵՐԳԸ,"ՍԱՆԴՌԻ ԵՐԳԸ\n\nՀէնդէք ընգա, սանդըռ կըդա, \nՍան...",,Dagldiyan_2023,unknown,65
165,ԳԱՐՋՈՒԳ ՄԱՌՏԻՆ,"ԳԱՐՋՈՒԳ ՄԱՌՏԻՆ\n\nՀէ՜յ, մէգդի գացէք մէյդան, \n...",,Dagldiyan_2023,unknown,150
166,ՑԱՓԷԹ,ՑԱՓԷԹ\n\nԿըլլայի – տա ան ատէնը էռսուն-էռսունհի...,,Dagldiyan_2023,unknown,482


In [ ]:
def tokenize(text):
    from re import split
    return split(r'\s+', text)

def preprocess(text):
      t = tokenize(text)
      words = [arm_remove_punct(word).lower() for word in t]
      return words

In [ ]:
df['preprocessed'] = df['text'].apply(preprocess)
df.head(3)

,title,text,date,source,genre,tokens_number,preprocessed
0,ԴՊՐՈՑԻ ՃԱՆՓԱՆ,ԴՊՐՈՑԻ ՃԱՆՓԱՆ \n(Թոփտիի խոսվածքով) \n\nԲուզլավ...,1968,Atamanyan_2006,poem,83,"[դպրոցի, ճանփան, թոփտիի, խոսվածքով, բուզլավուխ..."
1,ՄԵՐ «ԸՆԿԵՐԸ»,"ՄԵՐ «ԸՆԿԵՐԸ» \n\nԴասարանին դուռը բացվեցավ, Երկ...",1999,Atamanyan_2006,poem,66,"[մեր, ընկերը, դասարանին, դուռը, բացվեցավ, երկո..."
2,ՄԵՐ ԽԱՄՈՒԹԻ ՃԱՆՓԱՆ,ՄԵՐ ԽԱՄՈՒԹԻ ՃԱՆՓԱՆ \n(Թոփտիի խոսվածքով) \n\nՄե...,1971,Atamanyan_2006,poem,93,"[մեր, խամութի, ճանփան, թոփտիի, խոսվածքով, մեր,..."


In [ ]:
def get_unique_tokens(preprocessed_text):
    unique_tokens = set()
    for l in preprocessed_text:
        unique_tokens.update(l)
    return list(unique_tokens)

In [ ]:
unique_strings_list = get_unique_tokens(df['preprocessed'])
print(len(unique_strings_list))
print(unique_strings_list[:10])

22195
['', 'տեղերը', 'ագեր', 'չայնին', 'կավատ-կավատ', 'ծխոտը', 'կըբռնե', 'ապուլ', 'օշթանջա', 'էլլելեն']


### Import parsers

Eastern Armenian Uniparser

In [ ]:
!pip install uniparser-eastern-armenian

In [ ]:
from uniparser_eastern_armenian import EasternArmenianAnalyzer

In [ ]:
analyzer_EA = FasterAnalyzer(EasternArmenianAnalyzer())
analyzer_EA

In [ ]:
analyzer_EA.analyze_words('մարդ')

[{'wf': 'մարդ',
  'lemma': 'մարդ',
  'gramm': ['N', 'anim', 'hum', 'sg', 'nom', 'nonposs'],
  'wfGlossed': 'մարդ',
  'gloss': 'person',
  'trans_en': 'man, person, soul'}]

Western Armenian Uniparser

In [ ]:
!pip install uniparser_morph

In [ ]:
from uniparser_morph import Analyzer

Manually upload files archive

In [ ]:
!unzip 'western_uniparser' -d './western'

Archive:  western_uniparser.zip
replace ./western/lexemes/hyw-lexemes.txt? [y]es, [n]o, [A]ll, [N]one, [r]ename: N


In [ ]:
analyzer_WA = Analyzer()
analyzer_WA.lexFile = './western/lexemes/'
analyzer_WA.paradigmFile = './western/paradigms/'
analyzer_WA.load_grammar()

analyzer_WA = FasterAnalyzer(analyzer_WA)

In [ ]:
analyzer_WA.analyze_words('մարդ')

[{'wf': 'մարդ',
  'lemma': 'մարդ',
  'gramm': ['N', 'hum', 'anim', 'sg', 'nom'],
  'wfGlossed': 'մարդ',
  'gloss': 'STEM',
  'trans_en': 'man, person',
  'trans_fr': 'homme, humain'},
 {'wf': 'մարդ',
  'lemma': 'մար',
  'gramm': ['N', 'inanim', 'sg', 'nom', 'poss.2'],
  'wfGlossed': 'մար-դ',
  'gloss': 'STEM-2POSS',
  'trans_en': 'width, breath of linen; amphora',
  'trans_fr': 'mesure équivalant à une quarantaine de litres'},
 {'wf': 'մարդ',
  'lemma': 'մար',
  'gramm': ['N', 'inanim', 'sg', 'nom', 'poss.2'],
  'wfGlossed': 'մար-դ',
  'gloss': 'STEM-2POSS',
  'trans_en': '',
  'trans_fr': 'mède'}]

Customized Nor_Nakhichevan Armenian Uniparser

In [ ]:
# !pip install uniparser_morph
# from uniparser_morph import Analyzer

In [ ]:
!git clone 'https://github.com/NorNakhichevan-Armenian-corpus/uniparser-grammar-nornakhichevan-armenian.git'

fatal: destination path 'uniparser-grammar-nornakhichevan-armenian' already exists and is not an empty directory.


In [ ]:
analyzer_NNA = Analyzer()
analyzer_NNA.lexFile = './uniparser-grammar-nornakhichevan-armenian/lexemes/'
analyzer_NNA.paradigmFile = './uniparser-grammar-nornakhichevan-armenian/paradigms/'
analyzer_NNA.load_grammar()

analyzer_NNA = FasterAnalyzer(analyzer_NNA)

In [ ]:
analyzer_NNA.analyze_words('մարդ')

[{'wf': 'մարդ',
  'lemma': 'մարդ',
  'gramm': ['N', 'anim', 'hum', 'sg', 'nom', 'nonposs'],
  'wfGlossed': 'մարդ',
  'gloss': 'person',
  'trans_en': 'man, person, soul',
  'trans_ru': ''},
 {'wf': 'մարդ',
  'lemma': 'մար',
  'gramm': ['N', 'sg', 'nom', 'poss.2'],
  'wfGlossed': 'մար-դ',
  'gloss': 'STEM-2POSS',
  'trans_en': '',
  'trans_ru': 'мать'}]

### More definitions for benchmarking

In [ ]:
def make_analyses(preprocessed_text, analyzer):
    return [analyzer.analyze_words(w) for w in preprocessed_text]

In [ ]:
def count_parses(parses_list):
    # counts all cases when there is an analysis for all tokens in prepared list
    ct = 0
    for token_analyses in parses_list:
        for analysis in token_analyses:
            if analysis['lemma'] != '':
                ct += 1
    return ct

In [ ]:
def count_parses_total(parses_list):
    # counts the total number of analyses for all tokens in prepared list
    ct = 0
    for token_analyses in parses_list:
        for analysis in token_analyses:
            if analysis['lemma'] != '':
                # write down how many analyses there were
                ct += len(token_analyses)
    return ct

### Eastern Armenian Uniparser

In [ ]:
df['analyses_EA'] = df['preprocessed'].apply(make_analyses, args=(analyzer_EA,))
df.head(3)

,title,text,date,source,genre,tokens_number,preprocessed,analyses_EA
0,ԴՊՐՈՑԻ ՃԱՆՓԱՆ,ԴՊՐՈՑԻ ՃԱՆՓԱՆ \n(Թոփտիի խոսվածքով) \n\nԲուզլավ...,1968,Atamanyan_2006,poem,83,"[դպրոցի, ճանփան, թոփտիի, խոսվածքով, բուզլավուխ...","[[{'wf': 'դպրոցի', 'lemma': 'դպրոց', 'gramm': ..."
1,ՄԵՐ «ԸՆԿԵՐԸ»,"ՄԵՐ «ԸՆԿԵՐԸ» \n\nԴասարանին դուռը բացվեցավ, Երկ...",1999,Atamanyan_2006,poem,66,"[մեր, ընկերը, դասարանին, դուռը, բացվեցավ, երկո...","[[{'wf': 'մեր', 'lemma': 'մենք', 'gramm': ['PR..."
2,ՄԵՐ ԽԱՄՈՒԹԻ ՃԱՆՓԱՆ,ՄԵՐ ԽԱՄՈՒԹԻ ՃԱՆՓԱՆ \n(Թոփտիի խոսվածքով) \n\nՄե...,1971,Atamanyan_2006,poem,93,"[մեր, խամութի, ճանփան, թոփտիի, խոսվածքով, մեր,...","[[{'wf': 'մեր', 'lemma': 'մենք', 'gramm': ['PR..."


In [ ]:
df['count_EA'] = df['analyses_EA'].apply(count_parses)
df.head(3)

,title,text,date,source,genre,tokens_number,preprocessed,analyses_EA,count_EA,analyses_WA,count_WA,analyses_NNA,count_NNA
0,ԴՊՐՈՑԻ ՃԱՆՓԱՆ,ԴՊՐՈՑԻ ՃԱՆՓԱՆ \n(Թոփտիի խոսվածքով) \n\nԲուզլավ...,1968,Atamanyan_2006,poem,83,"[դպրոցի, ճանփան, թոփտիի, խոսվածքով, բուզլավուխ...","[[{'wf': 'դպրոցի', 'lemma': 'դպրոց', 'gramm': ...",46,"[[{'wf': 'դպրոցի', 'lemma': 'դպրոց', 'gramm': ...",53,"[[{'wf': 'դպրոցի', 'lemma': 'դպրոց', 'gramm': ...",66
1,ՄԵՐ «ԸՆԿԵՐԸ»,"ՄԵՐ «ԸՆԿԵՐԸ» \n\nԴասարանին դուռը բացվեցավ, Երկ...",1999,Atamanyan_2006,poem,66,"[մեր, ընկերը, դասարանին, դուռը, բացվեցավ, երկո...","[[{'wf': 'մեր', 'lemma': 'մենք', 'gramm': ['PR...",57,"[[{'wf': 'մեր', 'lemma': 'մենք', 'gramm': ['PR...",139,"[[{'wf': 'մեր', 'lemma': 'մենք', 'gramm': ['PR...",127
2,ՄԵՐ ԽԱՄՈՒԹԻ ՃԱՆՓԱՆ,ՄԵՐ ԽԱՄՈՒԹԻ ՃԱՆՓԱՆ \n(Թոփտիի խոսվածքով) \n\nՄե...,1971,Atamanyan_2006,poem,93,"[մեր, խամութի, ճանփան, թոփտիի, խոսվածքով, մեր,...","[[{'wf': 'մեր', 'lemma': 'մենք', 'gramm': ['PR...",62,"[[{'wf': 'մեր', 'lemma': 'մենք', 'gramm': ['PR...",136,"[[{'wf': 'մեր', 'lemma': 'մենք', 'gramm': ['PR...",102


In [ ]:
df['total_count_EA'] = df['analyses_EA'].apply(count_parses_total)
df.head(3)

,title,text,date,source,genre,tokens_number,preprocessed,analyses_EA,count_EA,analyses_WA,count_WA,analyses_NNA,count_NNA,total_count_EA
0,ԴՊՐՈՑԻ ՃԱՆՓԱՆ,ԴՊՐՈՑԻ ՃԱՆՓԱՆ \n(Թոփտիի խոսվածքով) \n\nԲուզլավ...,1968,Atamanyan_2006,poem,83,"[դպրոցի, ճանփան, թոփտիի, խոսվածքով, բուզլավուխ...","[[{'wf': 'դպրոցի', 'lemma': 'դպրոց', 'gramm': ...",46,"[[{'wf': 'դպրոցի', 'lemma': 'դպրոց', 'gramm': ...",53,"[[{'wf': 'դպրոցի', 'lemma': 'դպրոց', 'gramm': ...",66,54
1,ՄԵՐ «ԸՆԿԵՐԸ»,"ՄԵՐ «ԸՆԿԵՐԸ» \n\nԴասարանին դուռը բացվեցավ, Երկ...",1999,Atamanyan_2006,poem,66,"[մեր, ընկերը, դասարանին, դուռը, բացվեցավ, երկո...","[[{'wf': 'մեր', 'lemma': 'մենք', 'gramm': ['PR...",57,"[[{'wf': 'մեր', 'lemma': 'մենք', 'gramm': ['PR...",139,"[[{'wf': 'մեր', 'lemma': 'մենք', 'gramm': ['PR...",127,91
2,ՄԵՐ ԽԱՄՈՒԹԻ ՃԱՆՓԱՆ,ՄԵՐ ԽԱՄՈՒԹԻ ՃԱՆՓԱՆ \n(Թոփտիի խոսվածքով) \n\nՄե...,1971,Atamanyan_2006,poem,93,"[մեր, խամութի, ճանփան, թոփտիի, խոսվածքով, մեր,...","[[{'wf': 'մեր', 'lemma': 'մենք', 'gramm': ['PR...",62,"[[{'wf': 'մեր', 'lemma': 'մենք', 'gramm': ['PR...",136,"[[{'wf': 'մեր', 'lemma': 'մենք', 'gramm': ['PR...",102,96


In [ ]:
print(sum(df['count_EA']),
      round(sum(df['count_EA']) / sum(df['tokens_number']), 2))
print(sum(df['total_count_EA']),
      round(sum(df['total_count_EA']) / sum(df['tokens_number']), 2))

69511 0.68
132329 1.29


In [ ]:
unique_parses_EA = make_analyses(unique_strings_list, analyzer_EA)
print(len(unique_parses_EA))
print(unique_parses_EA[1])

22195
[{'wf': 'տեղերը', 'lemma': 'տեղ', 'gramm': ['N', 'inanim', 'pl', 'nom', 'def'], 'wfGlossed': 'տեղ-եր-ը', 'gloss': 'place-PL-DEF', 'trans_en': 'place, locality, location'}, {'wf': 'տեղերը', 'lemma': 'տեղ', 'gramm': ['A', 'oral', 'pl', 'nom', 'def'], 'wfGlossed': 'տեղ-եր-ը', 'gloss': 'STEM-PL-DEF', 'trans_en': ''}, {'wf': 'տեղերը', 'lemma': 'տեղ', 'gramm': ['PRON', 'ADV', 'oral', 'pl', 'nom', 'def'], 'wfGlossed': 'տեղ-եր-ը', 'gloss': 'there-PL-DEF', 'trans_en': 'there'}]


In [ ]:
print(count_parses(unique_parses_EA),
      round(count_parses(unique_parses_EA) / len(unique_parses_EA), 2))
print(count_parses_total(unique_parses_EA),
      round(count_parses_total(unique_parses_EA) / len(unique_parses_EA), 2))

9026 0.41
16248 0.73


### Wetern Armenian Uniparser

In [ ]:
df['analyses_WA'] = df['preprocessed'].apply(make_analyses, args=(analyzer_WA,))
df.head(3)

,title,text,date,source,genre,tokens_number,preprocessed,analyses_EA,count_EA,analyses_WA
0,ԴՊՐՈՑԻ ՃԱՆՓԱՆ,ԴՊՐՈՑԻ ՃԱՆՓԱՆ \n(Թոփտիի խոսվածքով) \n\nԲուզլավ...,1968,Atamanyan_2006,poem,83,"[դպրոցի, ճանփան, թոփտիի, խոսվածքով, բուզլավուխ...","[[{'wf': 'դպրոցի', 'lemma': 'դպրոց', 'gramm': ...",54,"[[{'wf': 'դպրոցի', 'lemma': 'դպրոց', 'gramm': ..."
1,ՄԵՐ «ԸՆԿԵՐԸ»,"ՄԵՐ «ԸՆԿԵՐԸ» \n\nԴասարանին դուռը բացվեցավ, Երկ...",1999,Atamanyan_2006,poem,66,"[մեր, ընկերը, դասարանին, դուռը, բացվեցավ, երկո...","[[{'wf': 'մեր', 'lemma': 'մենք', 'gramm': ['PR...",91,"[[{'wf': 'մեր', 'lemma': 'մենք', 'gramm': ['PR..."
2,ՄԵՐ ԽԱՄՈՒԹԻ ՃԱՆՓԱՆ,ՄԵՐ ԽԱՄՈՒԹԻ ՃԱՆՓԱՆ \n(Թոփտիի խոսվածքով) \n\nՄե...,1971,Atamanyan_2006,poem,93,"[մեր, խամութի, ճանփան, թոփտիի, խոսվածքով, մեր,...","[[{'wf': 'մեր', 'lemma': 'մենք', 'gramm': ['PR...",96,"[[{'wf': 'մեր', 'lemma': 'մենք', 'gramm': ['PR..."


In [ ]:
df['count_WA'] = df['analyses_WA'].apply(count_parses)
df.head(3)

,title,text,date,source,genre,tokens_number,preprocessed,analyses_EA,count_EA,analyses_WA,count_WA,analyses_NNA,count_NNA,total_count_EA
0,ԴՊՐՈՑԻ ՃԱՆՓԱՆ,ԴՊՐՈՑԻ ՃԱՆՓԱՆ \n(Թոփտիի խոսվածքով) \n\nԲուզլավ...,1968,Atamanyan_2006,poem,83,"[դպրոցի, ճանփան, թոփտիի, խոսվածքով, բուզլավուխ...","[[{'wf': 'դպրոցի', 'lemma': 'դպրոց', 'gramm': ...",46,"[[{'wf': 'դպրոցի', 'lemma': 'դպրոց', 'gramm': ...",39,"[[{'wf': 'դպրոցի', 'lemma': 'դպրոց', 'gramm': ...",66,54
1,ՄԵՐ «ԸՆԿԵՐԸ»,"ՄԵՐ «ԸՆԿԵՐԸ» \n\nԴասարանին դուռը բացվեցավ, Երկ...",1999,Atamanyan_2006,poem,66,"[մեր, ընկերը, դասարանին, դուռը, բացվեցավ, երկո...","[[{'wf': 'մեր', 'lemma': 'մենք', 'gramm': ['PR...",57,"[[{'wf': 'մեր', 'lemma': 'մենք', 'gramm': ['PR...",61,"[[{'wf': 'մեր', 'lemma': 'մենք', 'gramm': ['PR...",127,91
2,ՄԵՐ ԽԱՄՈՒԹԻ ՃԱՆՓԱՆ,ՄԵՐ ԽԱՄՈՒԹԻ ՃԱՆՓԱՆ \n(Թոփտիի խոսվածքով) \n\nՄե...,1971,Atamanyan_2006,poem,93,"[մեր, խամութի, ճանփան, թոփտիի, խոսվածքով, մեր,...","[[{'wf': 'մեր', 'lemma': 'մենք', 'gramm': ['PR...",62,"[[{'wf': 'մեր', 'lemma': 'մենք', 'gramm': ['PR...",72,"[[{'wf': 'մեր', 'lemma': 'մենք', 'gramm': ['PR...",102,96


In [ ]:
df['total_count_WA'] = df['analyses_WA'].apply(count_parses_total)
df.head(3)

,title,text,date,source,genre,tokens_number,preprocessed,analyses_EA,count_EA,analyses_WA,count_WA,analyses_NNA,count_NNA,total_count_EA,total_count_WA
0,ԴՊՐՈՑԻ ՃԱՆՓԱՆ,ԴՊՐՈՑԻ ՃԱՆՓԱՆ \n(Թոփտիի խոսվածքով) \n\nԲուզլավ...,1968,Atamanyan_2006,poem,83,"[դպրոցի, ճանփան, թոփտիի, խոսվածքով, բուզլավուխ...","[[{'wf': 'դպրոցի', 'lemma': 'դպրոց', 'gramm': ...",46,"[[{'wf': 'դպրոցի', 'lemma': 'դպրոց', 'gramm': ...",39,"[[{'wf': 'դպրոցի', 'lemma': 'դպրոց', 'gramm': ...",66,54,53
1,ՄԵՐ «ԸՆԿԵՐԸ»,"ՄԵՐ «ԸՆԿԵՐԸ» \n\nԴասարանին դուռը բացվեցավ, Երկ...",1999,Atamanyan_2006,poem,66,"[մեր, ընկերը, դասարանին, դուռը, բացվեցավ, երկո...","[[{'wf': 'մեր', 'lemma': 'մենք', 'gramm': ['PR...",57,"[[{'wf': 'մեր', 'lemma': 'մենք', 'gramm': ['PR...",61,"[[{'wf': 'մեր', 'lemma': 'մենք', 'gramm': ['PR...",127,91,139
2,ՄԵՐ ԽԱՄՈՒԹԻ ՃԱՆՓԱՆ,ՄԵՐ ԽԱՄՈՒԹԻ ՃԱՆՓԱՆ \n(Թոփտիի խոսվածքով) \n\nՄե...,1971,Atamanyan_2006,poem,93,"[մեր, խամութի, ճանփան, թոփտիի, խոսվածքով, մեր,...","[[{'wf': 'մեր', 'lemma': 'մենք', 'gramm': ['PR...",62,"[[{'wf': 'մեր', 'lemma': 'մենք', 'gramm': ['PR...",72,"[[{'wf': 'մեր', 'lemma': 'մենք', 'gramm': ['PR...",102,96,136


In [ ]:
print(sum(df['count_WA']),
      round(sum(df['count_WA']) / sum(df['tokens_number']), 2))
print(sum(df['total_count_WA']),
      round(sum(df['total_count_WA']) / sum(df['tokens_number']), 2))

70765 0.69
136411 1.33


In [ ]:
unique_parses_WA = make_analyses(unique_strings_list, analyzer_WA)
print(len(unique_parses_WA))
print(unique_parses_WA[1])

22195
[{'wf': 'տեղերը', 'lemma': 'տեղ', 'gramm': ['N', 'inanim', 'pl', 'nom', 'def'], 'wfGlossed': 'տեղ-եր-ը', 'gloss': 'STEM-DEF', 'trans_en': 'place', 'trans_fr': 'lieu'}, {'wf': 'տեղերը', 'lemma': 'տեղ', 'gramm': ['POST', 'pl', 'nom', 'def'], 'wfGlossed': 'տեղ-եր-ը', 'gloss': 'STEM-DEF', 'trans_en': 'place', 'trans_fr': ''}]


In [ ]:
print(count_parses(unique_parses_WA),
      round(count_parses(unique_parses_WA) / len(unique_parses_WA), 2))
print(count_parses_total(unique_parses_WA),
      round(count_parses_total(unique_parses_WA) / len(unique_parses_WA), 2))

8352 0.38
15630 0.7


### Nor-Nakhichevan Armenian Uniparser

In [ ]:
df['analyses_NNA'] = df['preprocessed'].apply(make_analyses, args=(analyzer_NNA,))
df.head(3)

,title,text,date,source,genre,tokens_number,preprocessed,analyses_EA,count_EA,analyses_WA,count_WA,analyses_NNA
0,ԴՊՐՈՑԻ ՃԱՆՓԱՆ,ԴՊՐՈՑԻ ՃԱՆՓԱՆ \n(Թոփտիի խոսվածքով) \n\nԲուզլավ...,1968,Atamanyan_2006,poem,83,"[դպրոցի, ճանփան, թոփտիի, խոսվածքով, բուզլավուխ...","[[{'wf': 'դպրոցի', 'lemma': 'դպրոց', 'gramm': ...",54,"[[{'wf': 'դպրոցի', 'lemma': 'դպրոց', 'gramm': ...",53,"[[{'wf': 'դպրոցի', 'lemma': 'դպրոց', 'gramm': ..."
1,ՄԵՐ «ԸՆԿԵՐԸ»,"ՄԵՐ «ԸՆԿԵՐԸ» \n\nԴասարանին դուռը բացվեցավ, Երկ...",1999,Atamanyan_2006,poem,66,"[մեր, ընկերը, դասարանին, դուռը, բացվեցավ, երկո...","[[{'wf': 'մեր', 'lemma': 'մենք', 'gramm': ['PR...",91,"[[{'wf': 'մեր', 'lemma': 'մենք', 'gramm': ['PR...",139,"[[{'wf': 'մեր', 'lemma': 'մենք', 'gramm': ['PR..."
2,ՄԵՐ ԽԱՄՈՒԹԻ ՃԱՆՓԱՆ,ՄԵՐ ԽԱՄՈՒԹԻ ՃԱՆՓԱՆ \n(Թոփտիի խոսվածքով) \n\nՄե...,1971,Atamanyan_2006,poem,93,"[մեր, խամութի, ճանփան, թոփտիի, խոսվածքով, մեր,...","[[{'wf': 'մեր', 'lemma': 'մենք', 'gramm': ['PR...",96,"[[{'wf': 'մեր', 'lemma': 'մենք', 'gramm': ['PR...",136,"[[{'wf': 'մեր', 'lemma': 'մենք', 'gramm': ['PR..."


In [ ]:
df['count_NNA'] = df['analyses_NNA'].apply(count_parses)
df.head(3)

,title,text,date,source,genre,tokens_number,preprocessed,analyses_EA,count_EA,analyses_WA,count_WA,analyses_NNA,count_NNA,total_count_EA,total_count_WA
0,ԴՊՐՈՑԻ ՃԱՆՓԱՆ,ԴՊՐՈՑԻ ՃԱՆՓԱՆ \n(Թոփտիի խոսվածքով) \n\nԲուզլավ...,1968,Atamanyan_2006,poem,83,"[դպրոցի, ճանփան, թոփտիի, խոսվածքով, բուզլավուխ...","[[{'wf': 'դպրոցի', 'lemma': 'դպրոց', 'gramm': ...",46,"[[{'wf': 'դպրոցի', 'lemma': 'դպրոց', 'gramm': ...",39,"[[{'wf': 'դպրոցի', 'lemma': 'դպրոց', 'gramm': ...",46,54,53
1,ՄԵՐ «ԸՆԿԵՐԸ»,"ՄԵՐ «ԸՆԿԵՐԸ» \n\nԴասարանին դուռը բացվեցավ, Երկ...",1999,Atamanyan_2006,poem,66,"[մեր, ընկերը, դասարանին, դուռը, բացվեցավ, երկո...","[[{'wf': 'մեր', 'lemma': 'մենք', 'gramm': ['PR...",57,"[[{'wf': 'մեր', 'lemma': 'մենք', 'gramm': ['PR...",61,"[[{'wf': 'մեր', 'lemma': 'մենք', 'gramm': ['PR...",67,91,139
2,ՄԵՐ ԽԱՄՈՒԹԻ ՃԱՆՓԱՆ,ՄԵՐ ԽԱՄՈՒԹԻ ՃԱՆՓԱՆ \n(Թոփտիի խոսվածքով) \n\nՄե...,1971,Atamanyan_2006,poem,93,"[մեր, խամութի, ճանփան, թոփտիի, խոսվածքով, մեր,...","[[{'wf': 'մեր', 'lemma': 'մենք', 'gramm': ['PR...",62,"[[{'wf': 'մեր', 'lemma': 'մենք', 'gramm': ['PR...",72,"[[{'wf': 'մեր', 'lemma': 'մենք', 'gramm': ['PR...",62,96,136


In [ ]:
df['total_count_NNA'] = df['analyses_NNA'].apply(count_parses_total)
df.head(3)

,title,text,date,source,genre,tokens_number,preprocessed,analyses_EA,count_EA,analyses_WA,count_WA,analyses_NNA,count_NNA,total_count_EA,total_count_WA,total_count_NNA
0,ԴՊՐՈՑԻ ՃԱՆՓԱՆ,ԴՊՐՈՑԻ ՃԱՆՓԱՆ \n(Թոփտիի խոսվածքով) \n\nԲուզլավ...,1968,Atamanyan_2006,poem,83,"[դպրոցի, ճանփան, թոփտիի, խոսվածքով, բուզլավուխ...","[[{'wf': 'դպրոցի', 'lemma': 'դպրոց', 'gramm': ...",46,"[[{'wf': 'դպրոցի', 'lemma': 'դպրոց', 'gramm': ...",39,"[[{'wf': 'դպրոցի', 'lemma': 'դպրոց', 'gramm': ...",46,54,53,66
1,ՄԵՐ «ԸՆԿԵՐԸ»,"ՄԵՐ «ԸՆԿԵՐԸ» \n\nԴասարանին դուռը բացվեցավ, Երկ...",1999,Atamanyan_2006,poem,66,"[մեր, ընկերը, դասարանին, դուռը, բացվեցավ, երկո...","[[{'wf': 'մեր', 'lemma': 'մենք', 'gramm': ['PR...",57,"[[{'wf': 'մեր', 'lemma': 'մենք', 'gramm': ['PR...",61,"[[{'wf': 'մեր', 'lemma': 'մենք', 'gramm': ['PR...",67,91,139,127
2,ՄԵՐ ԽԱՄՈՒԹԻ ՃԱՆՓԱՆ,ՄԵՐ ԽԱՄՈՒԹԻ ՃԱՆՓԱՆ \n(Թոփտիի խոսվածքով) \n\nՄե...,1971,Atamanyan_2006,poem,93,"[մեր, խամութի, ճանփան, թոփտիի, խոսվածքով, մեր,...","[[{'wf': 'մեր', 'lemma': 'մենք', 'gramm': ['PR...",62,"[[{'wf': 'մեր', 'lemma': 'մենք', 'gramm': ['PR...",72,"[[{'wf': 'մեր', 'lemma': 'մենք', 'gramm': ['PR...",62,96,136,102


In [ ]:
print(sum(df['count_NNA']),
      round(sum(df['count_NNA']) / sum(df['tokens_number']), 2))
print(sum(df['total_count_NNA']),
      round(sum(df['total_count_NNA']) / sum(df['tokens_number']), 2))

91534 0.89
244916 2.39


In [ ]:
unique_parses_NNA = make_analyses(unique_strings_list, analyzer_NNA)
print(len(unique_parses_NNA))
print(unique_parses_NNA[1])

22195
[{'wf': 'տեղերը', 'lemma': 'տեղ', 'gramm': ['N', 'inanim', 'pl', 'nom', 'def'], 'wfGlossed': 'տեղ-եր-ը', 'gloss': 'place-PL-DEF', 'trans_en': 'place, locality, location', 'trans_ru': 'место'}, {'wf': 'տեղերը', 'lemma': 'տեղ', 'gramm': ['PRON', 'ADV', 'oral', 'pl', 'nom', 'def'], 'wfGlossed': 'տեղ-եր-ը', 'gloss': 'there-PL-DEF', 'trans_en': 'there', 'trans_ru': ''}, {'wf': 'տեղերը', 'lemma': 'տեղ', 'gramm': ['A', 'oral', 'pl', 'nom', 'def'], 'wfGlossed': 'տեղ-եր-ը', 'gloss': 'STEM-PL-DEF', 'trans_en': '', 'trans_ru': ''}]


In [ ]:
print(count_parses(unique_parses_NNA),
      round(count_parses(unique_parses_NNA) / len(unique_parses_NNA), 2))
print(count_parses_total(unique_parses_NNA),
      round(count_parses_total(unique_parses_NNA) / len(unique_parses_NNA), 2))

8113 0.37
12463 0.56


In [ ]:
df.to_pickle('data2.pkl')

### Frequency analysis

In [22]:
from collections import Counter
from itertools import chain
from pprint import pprint

In [23]:
def get_token_frequencies(preprocessed_text):
    return Counter(chain.from_iterable(preprocessed_text))

In [24]:
token_counts = get_token_frequencies(df['preprocessed'])
print(len(token_counts))
print()
pprint(token_counts.most_common(302))

22195

[('ալ', 3081),
 ('', 2315),
 ('է', 1897),
 ('մեկ', 914),
 ('–', 904),
 ('ինչ', 894),
 ('ասաց', 764),
 ('իդա', 747),
 ('էր', 744),
 ('որ', 650),
 ('ես', 632),
 ('ադ', 610),
 ('պիտ', 518),
 ('և', 476),
 ('մեր', 463),
 ('նը', 454),
 ('շտե', 423),
 ('կը', 372),
 ('չէ', 364),
 ('մեչը', 354),
 ('բան', 344),
 ('հատ', 335),
 ('վրան', 309),
 ('համար', 296),
 ('ադպես', 295),
 ('ին', 285),
 ('սա', 269),
 ('կասե', 265),
 ('ան', 263),
 ('համա', 260),
 ('էղավ', 254),
 ('տունը', 251),
 ('ինձի', 249),
 ('հետ', 244),
 ('կենե', 233),
 ('մեմը', 230),
 ('ըռինդ', 223),
 ('պան', 221),
 ('խաթի', 218),
 ('ինչես', 217),
 ('պետք', 215),
 ('իմ', 213),
 ('աս', 211),
 ('դըհա', 211),
 ('նարա', 209),
 ('սաղ', 201),
 ('մաշդը', 199),
 ('մեծ', 194),
 ('էղած', 192),
 ('նա', 186),
 ('մաշդ', 183),
 ('քեզի', 181),
 ('հանա', 177),
 ('ասի', 177),
 ('էյ', 173),
 ('դուն', 171),
 ('չի', 164),
 ('քանի', 162),
 ('ինքն', 162),
 ('իրեն', 160),
 ('էին', 160),
 ('մեմ', 160),
 ('դա', 159),
 ('հայդը', 154),
 ('էլլավ', 153),
 ('է

In [25]:
text_size = 102592
token_wpm = sorted({
    k: round(v / text_size * 1000000) for k, v in token_counts.items() \
    if k != '' and k != '–'}.items(), key=lambda x: x[1], reverse=True)
print()
pprint(token_wpm[:300])


[('ալ', 30032),
 ('է', 18491),
 ('մեկ', 8909),
 ('ինչ', 8714),
 ('ասաց', 7447),
 ('իդա', 7281),
 ('էր', 7252),
 ('որ', 6336),
 ('ես', 6160),
 ('ադ', 5946),
 ('պիտ', 5049),
 ('և', 4640),
 ('մեր', 4513),
 ('նը', 4425),
 ('շտե', 4123),
 ('կը', 3626),
 ('չէ', 3548),
 ('մեչը', 3451),
 ('բան', 3353),
 ('հատ', 3265),
 ('վրան', 3012),
 ('համար', 2885),
 ('ադպես', 2875),
 ('ին', 2778),
 ('սա', 2622),
 ('կասե', 2583),
 ('ան', 2564),
 ('համա', 2534),
 ('էղավ', 2476),
 ('տունը', 2447),
 ('ինձի', 2427),
 ('հետ', 2378),
 ('կենե', 2271),
 ('մեմը', 2242),
 ('ըռինդ', 2174),
 ('պան', 2154),
 ('խաթի', 2125),
 ('ինչես', 2115),
 ('պետք', 2096),
 ('իմ', 2076),
 ('աս', 2057),
 ('դըհա', 2057),
 ('նարա', 2037),
 ('սաղ', 1959),
 ('մաշդը', 1940),
 ('մեծ', 1891),
 ('էղած', 1871),
 ('նա', 1813),
 ('մաշդ', 1784),
 ('քեզի', 1764),
 ('հանա', 1725),
 ('ասի', 1725),
 ('էյ', 1686),
 ('դուն', 1667),
 ('չի', 1599),
 ('քանի', 1579),
 ('ինքն', 1579),
 ('իրեն', 1560),
 ('էին', 1560),
 ('մեմ', 1560),
 ('դա', 1550),
 ('հայդը'

In [36]:
# retreived from http://eanc.net/EANC/stat/statistics.php?interface_language=en
EA_ranked_300 = '''է

եվ

որ

են

էր

ու

մի

ես

այդ

էլ

իր

թե

համար

նա

եմ

էին

այն

այս

ինչ

հետ

ոչ

բայց

իսկ

նրա

մեր

մեջ

չի

վրա

նաեվ

մասին

շատ

ա

ինչպես

կարող

երբ

հետո

կամ

իմ

ավելի

միայն

չէ

ինձ

ի

եթե

պետք

մեծ

իրենց

նրան

այլ

ենք

մենք

նրանց

քանի

ասաց

նոր

անգամ

ժամանակ

ամեն

որը

դա

բոլոր

մոտ

չէր

երկու

արդեն

մինչեվ

առաջին

չեն

լավ

1

առաջ

բան

ասում

դու

սակայն

2

մեկ

որոնք

հայ

դուրս

նրանք

հայաստանի

նման

չեմ

պիտի

եք

մեզ

ուզում

ըստ

քեզ

եղել

ապա

հիմա

կողմից

որի

ունի

տարի

դեռ

3

ձեր

քո

էի

ե

տալիս

իրեն

դուք

որպես

թ

ով

ինչու

հենց

կ

առանց

դեպի

մ

քաղաքական

որովհետեվ

տակ

դեմ

հ

մեկը

այստեղ

քիչ

կարելի

դեպքում

այսօր

գալիս

մարդ

օր

վերջին

կա

ասել

հհ

ամբողջ

նույն

ինքը

լինի

ձեզ

կը

որեվէ

այնպես

սա

տեղի

ազգային

ընթացքում

ուր

ս

նախագահ

հայկական

այնքան

դրա

պետական

4

մարդիկ

այժմ

ահա

երեկ

որտեղ

մյուս

եվս

քան

այդպես

միշտ

օրը

անում

գնում

ճիշտ

բարձր

տեղ

պես

նույնիսկ

ուրիշ

հա

երեք

տ

չկա

հազար

ինչ-որ

նորից

երկրի

նկատմամբ

գ

մարդկանց

վեր

միջազգային

այնտեղ

որոնց

այո

դ

տարբեր

մէջ

եկել

մարդու

թող

որն

թվում

հայաստանում

ոչինչ

ն

էս

ցույց

երկար

դրանք

կան

մը

երկրորդ

5

կարծես

էդ

10

վ

տեսնում

ինքն

փոքր

այսպես

տեր

որպեսզի

նայում

հայոց

հայտնի

պատճառով

գլխավոր

լ

լինում

տուն

ընդհանուր

եկավ

6

տարվա

կլինի

սովետական

ռ

դրամ

որոշ

լինել

ուրեմն

այսինքն

կարծում

ժողովրդի

օրինակ

որքան

բացի

ժամը

բաց

իրար

դե

աշխատանքի

աշխարհի

իհարկե

անհրաժեշտ

կարեվոր

չես

ում

թույլ

խոսում

միջեվ

հին

հնարավոր

տվեց

կյանքի

տալ

էինք

հանրապետության

դրանց

երեվանի

էլի

ունեն

ձեռք

ը

նույնպես

կուսակցության

հանկարծ

գործում

տվել

7

0

սիրում

ետ

այնպիսի

հիմնական

այլեվս

նախագահի

ծանր

հասարակական

պարզ

էն

հարցը

ունեցել

չենք

մասը

վերաբերյալ

ին

բոլորը

չէին

20

լինելու

այդպիսի

ազատ

չունի

8'''.replace('\n\n', '\n').split('\n')
print(len(EA_ranked_300))

300


In [37]:
set1 = set([i[0] for i in token_wpm[:300]])
set2 = set([i for i in EA_ranked_300])

In [38]:
print(len(set1.intersection(set2)))
pprint(set1.intersection(set2))

72
{'ա',
 'ամեն',
 'այդ',
 'այն',
 'այս',
 'անգամ',
 'առանց',
 'ասաց',
 'ավելի',
 'բայց',
 'բան',
 'դա',
 'դեպի',
 'են',
 'ես',
 'երբ',
 'է',
 'էի',
 'էին',
 'էինք',
 'էլ',
 'էր',
 'թ',
 'թող',
 'ժամանակ',
 'իմ',
 'ին',
 'ինչ',
 'ինքը',
 'ինքն',
 'իրեն',
 'իրենց',
 'կա',
 'կամ',
 'կան',
 'կը',
 'հա',
 'համար',
 'հետ',
 'ձեր',
 'մասին',
 'մեծ',
 'մեկ',
 'մեկը',
 'մեր',
 'մը',
 'մի',
 'միշտ',
 'նա',
 'նոր',
 'շատ',
 'ով',
 'որ',
 'որը',
 'որպեսզի',
 'ու',
 'ուրիշ',
 'չէ',
 'չէր',
 'չի',
 'պես',
 'պետք',
 'սա',
 'վրա',
 'տարի',
 'տեղ',
 'տուն',
 'փոքր',
 'քանի',
 'քիչ',
 'օր',
 'օրը'}


In [39]:
print(len(set1.difference(set2)))
pprint(set1.difference(set2))

228
{'ադ',
 'ադիվոր',
 'ադիվուկը',
 'ադպես',
 'ալ',
 'ալայ',
 'ալղը',
 'ախչիկանը',
 'ախչիկը',
 'ախպար',
 'ախռ',
 'աղա',
 'աղային',
 'աղան',
 'ան',
 'անել',
 'անելեն',
 'անելու',
 'անիմ',
 'անկից',
 'անունը',
 'անպես',
 'անցավ',
 'անքան',
 'աշտըխ',
 'առաչվան',
 'առավ',
 'առի',
 'աս',
 'ասա',
 'ասած',
 'ասելու',
 'ասի',
 'ասին',
 'ասլան',
 'ասպես',
 'ասօր',
 'արա',
 'արած',
 'արավ',
 'արի',
 'արիլ',
 'արին',
 'արինք',
 'բաբիյին',
 'բաբին',
 'բաներ',
 'բանը',
 'բանին',
 'բաշլայեցին',
 'բաշլայից',
 'բիդ',
 'բռնած',
 'գալաջի',
 'գեղի',
 'գենե',
 'գլոխը',
 'գնաց',
 'գոյա',
 'դեյեն',
 'դեյին',
 'դըհա',
 'դհա',
 'դուն',
 'դուս',
 'զայեր',
 'էդև',
 'էլլա',
 'էլլած',
 'էլլան',
 'էլլավ',
 'էկած',
 'էկան',
 'էկավ',
 'էկիլ',
 'էղա',
 'էղած',
 'էղածը',
 'էղավ',
 'էղիլ',
 'էյ',
 'էտեվ',
 'էտև',
 'էրկան',
 'էրկու',
 'ըլա',
 'ըլալու',
 'ըլար',
 'ընկավ',
 'ըռինդ',
 'թագավորին',
 'թիլքին',
 'թոփ',
 'թոփտի',
 'ժամանակը',
 'ժեք',
 'իդա',
 'իդապես',
 'իմիս',
 'ինա',
 'ինձի',
 'ինչես',
 'ինչի',
 'իս',
 'խաթի